# 12 - Data description
Read-only: computes the figures for the data description (unit of analysis, records, fields and types, target and class balance, time range, missingness, known limitations).
Nothing is written to BigQuery or the bucket.
Run after 10_split (and 08_weather_raw for the weather figures).

## Install pinned libraries
Read directly from the Github repo

In [ ]:
# Read from a git tag rather than from main, so a later re-run installs exactly the same versions.
CODE_VERSION = "code-v1"      # the git tag of the code being run: create it on GitHub before running
REPO = "katiedevane/tennis-project"
REQUIREMENTS_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

# Shared provenance helpers (code version, environment, index of data versions), from the same tag
PROVENANCE_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/provenance.py"
!wget -q -O provenance.py $PROVENANCE_URL

## Authenticate and setup
Enable Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"        # the bucket and the BigQuery datasets must share this location
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Record the code version and set the random seed
The git tag, commit, Python environment and random seed of this run are added to every manifest this notebook writes (see provenance.py).
`SEED` is the one project-wide seed: `start_run` applies it, and it should be passed wherever a function takes a seed (e.g. `random_state=SEED`).

In [ ]:
from provenance import start_run, register, table_version, sha256, utc_stamp, SEED

NOTEBOOK = "12_data_description"
RUN_INFO = start_run(f"https://github.com/{REPO}", CODE_VERSION, NOTEBOOK)

## Helpers

In [ ]:
import json
import pandas as pd
from google.cloud import bigquery, storage

bq = bigquery.Client(project=PROJECT_ID, location=REGION)
bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)   # safety cap: 1 GB per query
    return bq.query(sql, job_config=config).to_dataframe()

pd.set_option("display.max_rows", 200, "display.max_columns", 100, "display.width", 200)
RAW = f"{PROJECT_ID}.tennis_raw.wta_matches"
CLEAN = f"{PROJECT_ID}.tennis_clean.matches"
AB = f"{PROJECT_ID}.tennis_clean.matches_ab"
EDITIONS = f"{PROJECT_ID}.tennis_clean.tournament_editions"
LOCATIONS = f"{PROJECT_ID}.tennis_clean.venue_locations"
NAMEMAP = f"{PROJECT_ID}.tennis_clean.venue_name_map"
SPLITS = f"{PROJECT_ID}.tennis_features.match_splits"
LEVELS = "('G', 'PM', 'P', 'I')"          # the levels kept in 04_clean_matches

# The modelling table with its split: used by most cells below
MODEL = f"(SELECT a.*, s.split, s.modelling_ok FROM `{AB}` a JOIN `{SPLITS}` s USING (match_id))"
SPLIT_ORDER = "CASE split WHEN 'warmup' THEN 1 WHEN 'train' THEN 2 WHEN 'dev' THEN 3 WHEN 'test' THEN 4 ELSE 5 END"

## 1. Unit of analysis and number of records
Rows at each stage of the pipeline, from the raw archive to the rows used for fitting and evaluation.

In [ ]:
stages = run(f"""
SELECT 1 AS step, 'raw archive' AS stage, COUNT(*) AS matches, COUNT(DISTINCT tourney_id) AS tournament_ids FROM `{RAW}`
UNION ALL SELECT 2, 'in scope (tennis_clean.matches)', COUNT(*), COUNT(DISTINCT tourney_id) FROM `{CLEAN}`
UNION ALL SELECT 3, 'modelling rows (no walkovers)', COUNTIF(modelling_ok), COUNT(DISTINCT IF(modelling_ok, tourney_id, NULL)) FROM `{SPLITS}`
UNION ALL SELECT 4, 'fit and evaluation rows (not warmup)', COUNTIF(modelling_ok AND split != 'warmup'),
                 COUNT(DISTINCT IF(modelling_ok AND split != 'warmup', tourney_id, NULL)) FROM `{SPLITS}`
ORDER BY step""")
stages["removed"] = (-stages["matches"].diff()).astype("Int64")
stages

### Why raw rows were excluded

In [ ]:
run(f"""
WITH rr AS (SELECT DISTINCT tourney_id FROM `{RAW}` WHERE round = 'RR')
SELECT CASE WHEN tourney_level NOT IN {LEVELS} THEN CONCAT('level ', tourney_level)
            WHEN tourney_id IN (SELECT tourney_id FROM rr) THEN 'event with a round-robin stage'
            ELSE 'kept' END AS reason,
       COUNT(*) AS matches, COUNT(DISTINCT tourney_id) AS tournament_ids,
       STRING_AGG(DISTINCT tourney_name, ', ' ORDER BY tourney_name LIMIT 4) AS example_names
FROM `{RAW}` GROUP BY reason ORDER BY matches DESC""")

### Entities behind the matches

In [ ]:
run(f"""
SELECT
  (SELECT COUNT(*) FROM `{EDITIONS}`) AS tournament_editions,
  (SELECT COUNT(*) FROM `{NAMEMAP}`) AS tournament_names,
  (SELECT COUNT(*) FROM `{LOCATIONS}`) AS venues,
  (SELECT COUNT(DISTINCT id) FROM (SELECT a_id AS id FROM `{AB}` UNION ALL SELECT b_id FROM `{AB}`)) AS players,
  (SELECT APPROX_QUANTILES(n, 2)[OFFSET(1)] FROM (SELECT COUNT(*) AS n FROM `{AB}` GROUP BY tourney_id)) AS median_matches_per_edition
""").T

## 2. Fields and data types
Column count and columns per type for every table. The full list with descriptions and % missing per column is produced by 11_data_dictionary.

In [ ]:
overview = []
for t in [RAW, CLEAN, f"{PROJECT_ID}.tennis_clean.matches_dated", AB, SPLITS, EDITIONS, LOCATIONS]:
    schema = bq.get_table(t).schema
    types = pd.Series([f.field_type for f in schema]).value_counts().to_dict()
    overview.append({"table": t.split(".", 1)[1], "columns": len(schema), **types})
pd.DataFrame(overview).fillna(0).set_index("table").astype(int)

### Columns of the modelling table, grouped
`post_` columns are only known after the match.

In [ ]:
schema = pd.DataFrame([{"column": f.name, "type": f.field_type} for f in bq.get_table(AB).schema])
def group(c):
    if c == "a_won": return "target"
    if c.startswith("post_"): return "post-match (history only)"
    if c.startswith(("a_", "b_")): return "player attributes (A and B)"
    if c in ("match_id", "tourney_id"): return "identifiers"
    if c.startswith("match_date"): return "estimated match day"
    return "tournament context"
schema["group"] = schema["column"].map(group)
schema.groupby(["group", "type"]).agg(n=("column", "count"), columns=("column", ", ".join))

## 3. Target variable and class balance
Target `a_won` = 1 if player A won. Shown per split for the modelling rows, with the share won by the better-ranked player (the baseline to beat).

In [ ]:
balance = run(f"""
SELECT split, MIN(season) AS first_season, MAX(season) AS last_season,
       COUNT(*) AS matches, COUNTIF(modelling_ok) AS modelling_rows,
       ROUND(AVG(IF(modelling_ok, a_won, NULL)), 4) AS share_a_won,
       ROUND(AVG(IF(modelling_ok AND a_rank != b_rank, IF((a_rank < b_rank) = (a_won = 1), 1, 0), NULL)), 4) AS better_ranked_won,
       COUNTIF(modelling_ok AND a_rank IS NOT NULL AND b_rank IS NOT NULL AND a_rank != b_rank) AS rows_with_both_ranks
FROM {MODEL} GROUP BY split ORDER BY MIN({SPLIT_ORDER})""")
balance

In [ ]:
run(f"SELECT a_won, COUNT(*) AS n_rows, ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct FROM {MODEL} WHERE modelling_ok GROUP BY a_won ORDER BY a_won")

## 4. Time range and splits

In [ ]:
run(f"""
SELECT split, MIN(tourney_date) AS first_tournament_start, MAX(tourney_date) AS last_tournament_start,
       MAX(match_date_max) AS last_possible_match_day, COUNT(DISTINCT tourney_id) AS editions, COUNT(*) AS matches
FROM {MODEL} GROUP BY split ORDER BY MIN({SPLIT_ORDER})""")

### Matches and editions per season

In [ ]:
per_season = run(f"""
SELECT season, ANY_VALUE(split) AS split, COUNT(DISTINCT tourney_id) AS editions, COUNT(*) AS matches,
       COUNTIF(tourney_level = 'G') AS slam, COUNTIF(tourney_level = 'PM') AS premier_mandatory,
       COUNTIF(tourney_level = 'P') AS premier, COUNTIF(tourney_level = 'I') AS international
FROM {MODEL} GROUP BY season ORDER BY season""")
per_season

### Surface, event format and best-of

In [ ]:
print(run(f"SELECT surface, COUNT(*) AS matches, ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct FROM {MODEL} GROUP BY 1 ORDER BY 2 DESC"))
print(run(f"SELECT event_format, COUNT(DISTINCT tourney_id) AS editions, COUNT(*) AS matches FROM {MODEL} GROUP BY 1 ORDER BY 2 DESC"))
print(run(f"SELECT best_of, COUNT(*) AS matches FROM {MODEL} GROUP BY 1"))

## 5. Missingness
% of modelling rows with the value missing, by split. For player fields, a row counts as missing when either player lacks the value. Seed and entry are empty for unseeded and direct-entry players, so they are not true missing values.

In [ ]:
groups = {
    "rank": "a_rank IS NULL OR b_rank IS NULL",
    "rank_points": "a_rank_points IS NULL OR b_rank_points IS NULL",
    "height": "a_ht IS NULL OR b_ht IS NULL",
    "age": "a_age IS NULL OR b_age IS NULL",
    "hand_missing_or_U": "a_hand IS NULL OR b_hand IS NULL OR a_hand = 'U' OR b_hand = 'U'",
    "seed_A_structural": "a_seed IS NULL",
    "entry_A_structural": "a_entry IS NULL",
    "surface": "surface IS NULL",
    "minutes_post": "post_minutes IS NULL",
    "serve_stats_post": "post_a_svpt IS NULL OR post_b_svpt IS NULL",
}
cols = ",\n  ".join(f"ROUND(100 * COUNTIF({cond}) / COUNT(*), 1) AS {name}" for name, cond in groups.items())
missing = run(f"""
SELECT split, COUNT(*) AS n_rows, {cols} FROM {MODEL} WHERE modelling_ok GROUP BY split
UNION ALL
SELECT 'all', COUNT(*), {cols} FROM {MODEL} WHERE modelling_ok
""").set_index("split").T
missing[[c for c in ["warmup", "train", "dev", "test", "holdout_2026", "all"] if c in missing.columns]]

### Missingness by season for the fields that change over time

In [ ]:
run(f"""
SELECT season,
       ROUND(100 * COUNTIF(a_ht IS NULL OR b_ht IS NULL) / COUNT(*), 1) AS pct_no_height,
       ROUND(100 * COUNTIF(post_minutes IS NULL) / COUNT(*), 1) AS pct_no_minutes,
       ROUND(100 * COUNTIF(post_a_svpt IS NULL OR post_b_svpt IS NULL) / COUNT(*), 1) AS pct_no_serve_stats,
       ROUND(100 * COUNTIF(a_rank IS NULL OR b_rank IS NULL) / COUNT(*), 1) AS pct_no_rank
FROM {MODEL} WHERE modelling_ok GROUP BY season ORDER BY season""")

### Weather: completeness of the stored records
Read from the latest weather manifest (written by 08_weather_raw).

In [ ]:
names = sorted(b.name for b in bucket.list_blobs(prefix="manifests/open_meteo_archive_v2_"))
m = json.loads(bucket.blob(names[-1]).download_as_text())
w = pd.DataFrame(m.get("records") or m["files"])          # "records" in the revised manifest, "files" in the earlier one
editions = run(f"SELECT tourney_id FROM `{EDITIONS}`")
print("manifest:", names[-1])
print("editions:", len(editions), "| with weather:", w["tourney_id"].isin(editions["tourney_id"]).sum(),
      "| hourly rows:", int(w["n_hours"].sum()), "| null values:", int(w["null_values"].sum()))
print("window length (days):", (w["n_hours"] / 24).describe()[["min", "mean", "max"]].round(1).to_dict())
print("period:", w["start_date"].min(), "to", w["end_date"].max())

## 6. Known limitations: supporting figures

### Match date is not recorded
Every tournament has one `tourney_date` (its start); the match day is estimated as a window. Window width in days:

In [ ]:
run(f"""
SELECT event_format, DATE_DIFF(match_date_max, match_date_min, DAY) + 1 AS window_days, COUNT(*) AS matches
FROM {MODEL} GROUP BY 1, 2 ORDER BY 1, 2""")

### Results that are not a full match

In [ ]:
run(f"""
SELECT split, COUNTIF(post_is_walkover) AS walkovers, COUNTIF(post_is_retirement) AS retirements,
       COUNTIF(post_is_default) AS defaults,
       ROUND(100 * COUNTIF(post_is_retirement) / COUNTIF(modelling_ok), 1) AS pct_retired_of_modelling_rows
FROM {MODEL} GROUP BY split ORDER BY MIN({SPLIT_ORDER})""")

### Players with little history (cold start)
Matches per player across all modelling rows, and, per split, the share of player appearances with fewer than 10 matches in earlier tournaments (2010 counts from zero).

In [ ]:
print(run(f"""
WITH p AS (SELECT id, COUNT(*) AS n FROM (SELECT a_id AS id FROM {MODEL} WHERE modelling_ok
                                         UNION ALL SELECT b_id FROM {MODEL} WHERE modelling_ok) GROUP BY id)
SELECT COUNT(*) AS players, APPROX_QUANTILES(n, 2)[OFFSET(1)] AS median_matches,
       COUNTIF(n < 10) AS players_under_10_matches, COUNTIF(n >= 100) AS players_100_plus
FROM p"""))

run(f"""
WITH long AS (
  SELECT match_id, split, modelling_ok, tourney_date, a_id AS id FROM {MODEL}
  UNION ALL SELECT match_id, split, modelling_ok, tourney_date, b_id FROM {MODEL}),
prior AS (
  SELECT match_id, split, modelling_ok,
         COUNT(*) OVER (PARTITION BY id ORDER BY UNIX_DATE(tourney_date) RANGE BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS earlier
  FROM long)
SELECT split, ROUND(100 * AVG(IF(earlier < 10, 1, 0)), 1) AS pct_players_with_under_10_matches_in_earlier_tournaments
FROM prior WHERE modelling_ok GROUP BY split ORDER BY MIN({SPLIT_ORDER})""")

### Category values to be aware of

In [ ]:
print(run(f"SELECT season, tourney_name, COUNT(*) AS matches FROM {MODEL} WHERE surface = 'carpet' GROUP BY 1, 2 ORDER BY 1"))
print(run(f"SELECT hand, COUNT(*) AS player_sides FROM (SELECT a_hand AS hand FROM {MODEL} UNION ALL SELECT b_hand FROM {MODEL}) GROUP BY 1 ORDER BY 2 DESC"))
print(run(f"SELECT entry, COUNT(*) AS player_sides FROM (SELECT a_entry AS entry FROM {MODEL} UNION ALL SELECT b_entry FROM {MODEL}) GROUP BY 1 ORDER BY 2 DESC"))
print(run(f"SELECT LEAST(MIN(a_rank), MIN(b_rank)) AS best_rank, GREATEST(MAX(a_rank), MAX(b_rank)) AS worst_rank FROM {MODEL}"))

### Weather windows cut back for the archive delay, and the last season

In [ ]:
print(run(f"SELECT COUNTIF(end_date <= last_match_day) AS editions_window_cut_back, MAX(end_date) AS last_weather_day FROM `{EDITIONS}`"))
run(f"SELECT MAX(tourney_date) AS last_2026_tournament_start, COUNT(*) AS matches_2026 FROM {MODEL} WHERE season = 2026")